# Fundamentos Prácticos de Aprendizaje Automático

Este notebook contiene el resumen de los fundamentos de Machine Learning, incluyendo la teoría, el flujo de trabajo y la implementación práctica con Python y la biblioteca `scikit-learn`.


## 1. Introducción al Aprendizaje Automático

El Aprendizaje Automático (Machine Learning, ML) es una rama de la Inteligencia Artificial (IA)
que permite a los sistemas aprender de los datos y mejorar su rendimiento sin ser programados
explícitamente.

### Tipos principales:

- **Supervisado :** Se entrena con datos *etiquetados*.
  - Ejemplos: Predecir si un correo es spam (Clasificación), Predecir un precio (Regresión).
- **No supervisado :** Se entrena con datos *sin etiquetas*, buscando patrones.
  - Ejemplos: Segmentación de clientes (Clustering), Proyección de muchas variables en pocos componentes (reducción de dimensionalidad).
- **Por refuerzo :** El agente aprende por prueba y error, maximizando una recompensa.
  - Ejemplos: Entrenamiento de robots, IA para juegos.

## Flujo de trabajo
1. Defina el problema y la variable objetivo.
2. Separe entrenamiento, validación y prueba.
3. Ajuste el preprocesamiento solamente con entrenamiento.
4. Entrene y seleccione modelos con validación.
5. Evalúe una vez con prueba. El despliegue es una etapa posterior, fuera de este ejemplo.

In [ ]:
# Ejemplo Práctico en Python: División de Datos (Train y Test)

# Importaciones necesarias
import pandas as pd
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split

# Cargar un dataset de ejemplo: Iris
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target

print(f"Dataset Iris cargado. Características (X): {X.shape}, Target (y): {y.shape}")

# División inicial: 70% entrenamiento, 30% prueba
# Usamos 'random_state' para reproducibilidad.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y # 'stratify' para preservar las proporciones de las clases
)

print("\n--- División Train/Test (70% / 30%) ---")
print("Tamaño entrenamiento (X_train):", X_train.shape)
print("Tamaño prueba (X_test):", X_test.shape)


## 3. Métricas de Evaluación (Clasificación)

Para problemas de **Clasificación**, métricas comunes son:
- **Accuracy:** Proporción de aciertos totales.
- **Precision:** Cuántos de los positivos predichos son correctos (Evita Falsos Positivos).
- **Recall (Sensibilidad):** Cuántos de los positivos reales detectamos (Evita Falsos Negativos).
- **F1-Score:** Media armónica de Precision y Recall.

In [ ]:
# Ejemplo Práctico: Entrenamiento y Evaluación del Modelo usando un Modelo de Regresión Logística

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Selección y Entrenamiento del modelo
model = LogisticRegression(max_iter=200, random_state=42)
# El 'fit' (entrenamiento) solo usa los datos de entrenamiento
model.fit(X_train, y_train)

# 2. Predicción en el conjunto de prueba
y_pred = model.predict(X_test)

# 3. Evaluación
# Usamos 'average='macro'' para clasificaciones multiclase como la de Iris.
print("\n--- Métricas de Clasificación ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")


## Promedios de métricas multiclase
La siguiente sección compara macro, micro y weighted. Mantenga el mismo promedio al comparar modelos.



---
Promedios (`average`) en Métricas de Clasificación

El parámetro `average` en métricas de evaluación (*Precision*, *Recall*, *F1-Score*) de librerías como **Scikit-Learn** determina cómo se agregan los resultados entre las diferentes clases.

---

## 1. `average='binary'`

### **Cuándo usarlo:**

* Exclusivamente en **clasificación binaria** (solo existen 2 clases: `0` y `1`, o `Negativo` y `Positivo`).
* Cuando existe una clase específica de interés (**clase positiva**) y deseas evaluar el rendimiento centrándote únicamente en detectar esa clase.

### **Casos de uso típicos:**

* **Detección de fraude:** Interesa evaluar la clase `1: Fraude`.
* **Diagnóstico médico:** Interesa evaluar la clase `1: Enfermo`.
* **Filtro de correo:** Interesa evaluar la clase `1: Spam`.

---

## 2. `average='micro'`

### **Cuándo usarlo:**

* En problemas de **clasificación multiclase o multietiqueta**.
* Cuando hay un **desbalance severo de clases** y te importa el **rendimiento global por instancia**, asignándole el mismo peso a cada muestra sin importar a qué clase pertenece.
* Cuando deseas evaluar la precisión global de todas las predicciones combinadas.

### **Características principales:**

* Le da mayor peso a las clases mayoritarias (aporta más muestras al cálculo total de Verdaderos Positivos, Falsos Positivos y Falsos Negativos).
* En clasificación multiclase con una sola etiqueta por muestra, *Precision*, *Recall* y *F1-Score* calculados con `micro` coinciden numéricamente con el **Accuracy**.

### **Casos de uso típicos:**

* Categorización de artículos de e-commerce donde unas pocas categorías concentran la mayoría de productos, pero te interesa evaluar el desempeño general sobre todo el flujo de ventas.

---

## 3. `average='macro'`

### **Cuándo usarlo:**

* En problemas de **clasificación multiclase**.
* Cuando **todas las clases son igualmente importantes**, independientemente del número de instancias que contenga cada una.
* Cuando deseas detectar si el modelo falla o se comporta mal en **clases minoritarias (raras)**.

### **Características principales:**

* Trata a todas las clases por igual asignando a cada una el mismo peso ($1/N$).
* Si el modelo falla por completo en una clase minoritaria, su puntuación recibe el mismo peso que la de cualquier otra clase. El impacto total depende del número de clases.

### **Casos de uso típicos:**

* **Clasificación de enfermedades raras:** Identificar correctamente una condición poco frecuente es tan vital como diagnosticar la condición más común.
* **Reconocimiento de caracteres (OCR):** Cada dígito del `0` al `9` debe identificarse con la misma importancia.

---

## 4. `average='weighted'`

### **Cuándo usarlo:**

* En problemas de **clasificación multiclase** con **clases desbalanceadas**.
* Cuando quieres evaluar cada clase de forma independiente pero ajustando la importancia final al **soporte (número de instancias reales)** de cada clase en el dataset.

### **Características principales:**

* A diferencia de `macro`, asigna mayor peso a las clases que tienen más presencia en el conjunto de datos de prueba.
* Es una media de métricas por clase ponderada por soporte; puede ocultar resultados deficientes en clases minoritarias.

### **Casos de uso típicos:**

* **Clasificación de intenciones en Chatbots:** Las consultas habituales ocurren con mayor frecuencia en producción, pero requieres un valor ponderado según el tráfico real.

---

## Resumen de decisión rápida

| Escenario / Necesidad | Promedio recomendado |
| --- | --- |
| Evaluación sobre 2 clases enfocado en la clase positiva. | `average='binary'` |
| Múltiples clases desbalanceadas; se busca ponderar el peso según la frecuencia real de cada clase (soporte). | `average='weighted'` |
| Múltiples clases; se busca dar exactamente la **misma prioridad** a clases raras y comunes. | `average='macro'` |
| Múltiples clases o multietiqueta; se evalúa el **desempeño global por muestra individual**. | `average='micro'` |

## Métricas para Regresión

Para problemas de **Regresión** (predecir un valor continuo), usamos:
- MAE (Mean Absolute Error)
- MSE (Mean Squared Error)
- R² (Coeficiente de determinación)

## Ejercicios Propuestos

¡Hora de practicar! Tomen el reto e investiga lo listado de las siguientes celdas:

1. **Diferencia Conceptual:** Explica con tus propias palabras la diferencia
   entre aprendizaje supervisado y no supervisado.

2. **División Múltiple:** Usa `train_test_split` para dividir el dataset
   `wine` de sklearn en 60% entrenamiento, 20% validación y 20% prueba.
   *(Pista: Deberán usar `train_test_split` dos veces.)*

3. **Nuevo Modelo:** Entrena un modelo de clasificación con `RandomForestClassifier`
   (de `sklearn.ensemble`) usando los datos de entrenamiento de Iris (X_train, y_train)
   y calcula sus métricas en el conjunto de prueba (X_test, y_test).

In [ ]:
# Celda para el ejercicio 1:
# Escribe tu respuesta aquí.

In [ ]:
# Celda para el ejercicio 2: División Múltiple (Wine Dataset)
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

# 1. Cargar el dataset Wine
wine = load_wine(as_frame=True)
X_wine, y_wine = wine.data, wine.target

# 2. Dividir en Entrenamiento (60%) y Temporal (40%)
# X_train_60, X_temp_40, y_train_60, y_temp_40 = ...

# 3. Dividir Temporal (40%) en Validación (20%) y Prueba (20%)
# X_val_20, X_test_20, y_val_20, y_test_20 = ... (usa test_size=0.5)

# print("Tamaños finales:...")

In [ ]:
# Celda para el ejercicio 3: RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Importar el modelo

# 2. Inicializar y Entrenar (usando X_train e y_train del ejemplo Iris)

# 3. Predecir en X_test

# 4. Calcular y mostrar las métricas